In [2]:
import pandas as pd 
df= pd.read_csv("../data/rice_gene_database.csv")
print(df.shape)
df.head()

(35806, 12)


,Gene_ID,Chromosome,Start,End,Strand,Biotype,Protein_Sequence,Protein_Length,Transcript_ID,Description,GO_ID,GO_Term
0,Os01g0100100,1,2983,10815,+,protein_coding,MSSAAGQDNGDTAGDYIKWMCGAGGRAGGAMANLQRGVGSLVRDIG...,702.0,Os01t0100100-01,NaN,GO:0005096,GTPase activator activity
1,Os01g0100200,1,11218,12435,+,protein_coding,MEEAGERDADETHAWSGTASPAALWKTVASSAAMLKLALAMISAAF...,142.0,Os01t0100200-01,NaN,NaN,NaN
2,Os01g0100300,1,11372,12284,-,protein_coding,VCQLAHYIVTAQPHAHVRERPRHGRGRRVGSPPLHRRPCLLRHQPQ...,269.0,Os01t0100300-00,NaN,GO:0005506; GO:0020037; GO:0016705; GO:0004497...,iron ion binding; heme binding; oxidoreductase...
3,Os01g0100400,1,12721,15685,+,protein_coding,MDSIRRRSAGGILGILFLVLLRWAGAGDPYAYYEWEVSYVWGAPLG...,593.0,Os01t0100400-01,NaN,GO:0016491; GO:0005507,oxidoreductase activity; copper ion binding
4,Os01g0100466,1,12808,13978,-,protein_coding,MPQFVPPTPSCQGLLRCCTPCHVSSSGSSRPFLTFTTRFQLVVTFS...,77.0,Os01t0100466-00,NaN,NaN,NaN


In [3]:
#check any missing values
df.isna().sum()

Gene_ID                 0
Chromosome              0
Start                   0
End                     0
Strand                  0
Biotype                 0
Protein_Sequence        2
Protein_Length          2
Transcript_ID         111
Description         35701
GO_ID               14216
GO_Term             14223
dtype: int64

In [4]:
print(df["GO_Term"].notna().sum())
print(df["GO_Term"].isna().sum())


21583
14223


In [5]:
df[df["GO_Term"].isna()].head()

,Gene_ID,Chromosome,Start,End,Strand,Biotype,Protein_Sequence,Protein_Length,Transcript_ID,Description,GO_ID,GO_Term
1,Os01g0100200,1,11218,12435,+,protein_coding,MEEAGERDADETHAWSGTASPAALWKTVASSAAMLKLALAMISAAF...,142.0,Os01t0100200-01,NaN,NaN,NaN
4,Os01g0100466,1,12808,13978,-,protein_coding,MPQFVPPTPSCQGLLRCCTPCHVSSSGSSRPFLTFTTRFQLVVTFS...,77.0,Os01t0100466-00,NaN,NaN,NaN
5,Os01g0100500,1,16399,20144,+,protein_coding,MACTAARMFASNATLCACEPGFYLSAAINGTCLGLPDGGWQVGSVG...,524.0,Os01t0100500-01,NaN,NaN,NaN
7,Os01g0100650,1,25861,26424,-,protein_coding,PTLHLPSQLDSYLPFRTGPSLPSTPGSRRACTNILFAAPTCSLFSS...,127.0,Os01t0100650-00,NaN,NaN,NaN
9,Os01g0100800,1,29818,34453,+,protein_coding,MVLGKIVIVIGSGIVGTLLTSGEAKIALPDFRDVLSGAFKFVTKQD...,356.0,Os01t0100800-01,NaN,NaN,NaN


In [ ]:
# Replace missing GO annotations (NaN) with readable text.
# Sentence Transformer models understand normal language better than missing values.
# This also prevents errors when we combine multiple text columns.

In [6]:
df["GO_Term"] = df["GO_Term"].fillna("No GO annotation")

In [ ]:
# Replace missing gene descriptions with placeholder text.
# Many rice genes do not have curated descriptions in BioMart.
# Instead of leaving them as NaN, we explicitly tell the model that no description exists.

In [7]:
df["Description"] = df["Description"].fillna("No description")

In [ ]:
# Create one searchable text document for every gene.
# Instead of embedding each column separately, we combine the most informative
# biological information into one sentence. This becomes the input for the
# Sentence Transformer model, which converts it into a semantic embedding.

In [10]:
df["Search_Text"] = (
    "Gene ID: " + df["Gene_ID"] +
    ". Description: " + df["Description"] +
    ". GO Terms: " + df["GO_Term"] +
    ". Biotype: " + df["Biotype"] +
    ". Protein length: " + df["Protein_Length"].fillna(0).astype(int).astype(str) + " amino acids."
)

In [9]:
print(df["Protein_Length"].describe())

count    35804.000000
mean       323.597056
std        249.198281
min          2.000000
25%        138.000000
50%        259.000000
75%        440.000000
max       5342.000000
Name: Protein_Length, dtype: float64


In [ ]:
Rice Gene Database
        │
        ▼
Create Search_Text  ← You are here
        │
        ▼
Generate Embeddings
        │
        ▼
Vector Database (FAISS)
        │
        ▼
Retrieve relevant genes
        │
        ▼
LLM generates an answer

In [11]:
from sentence_transformers import SentenceTransformer

In [21]:
# ==========================================
# Load Sentence Transformer model
# This model converts gene descriptions into
# numerical embeddings for semantic search.
# ==========================================

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


In [ ]:
rice_gene_database.csv
          │
          ▼
Create Search_Text
          ✅ Done
          │
          ▼
Generate Embeddings
          ⬅️ TODAY
          │
          ▼
Build FAISS Vector Database
          │
          ▼
Ask Questions
          │
          ▼
Retrieve Similar Genes
          │
          ▼
Connect an LLM
          │
          ▼
PlantGPT 🌱

In [22]:
df.loc[0, "Search_Text"]

'Gene ID: Os01g0100100. Description: No description. GO Terms: GTPase activator activity. Biotype: protein_coding. Protein length: 702 amino acids.'

In [23]:
embedding = model.encode(df.loc[0, "Search_Text"])

print(type(embedding))
print(embedding.shape)
print(embedding.dtype)
print(embedding[:10])

<class 'numpy.ndarray'>
(384,)
float32
[-0.04091075 -0.09050086 -0.0650701   0.01952336  0.00404755 -0.02854167
  0.03894223  0.05863048 -0.05234058 -0.04913278]


In [ ]:
#we'll start w encoding 5 gene

In [24]:
sample_text=df["Search_Text"].head(5).tolist()

embeddings= model.encode(sample_text)

print(type(embeddings))
print(embeddings.shape)

<class 'numpy.ndarray'>
(5, 384)


In [ ]:
#.tolist() is used bcz Many AI libraries expect a normal Python list. so we conver the pandas series into list

In [ ]:
#chunking the data

In [38]:
import numpy as np

In [39]:
all_embeddings = []

chunk_size = 500

for i in range(0, len(df), chunk_size):

    texts = df["Search_Text"].iloc[i:i+chunk_size].tolist()

    emb = model.encode(
        texts,
        batch_size=32,
        show_progress_bar=False
    )

    all_embeddings.append(emb)

    print(f"Done {i+len(texts)} / {len(df)}")

Done 500 / 35806
Done 1000 / 35806
Done 1500 / 35806
Done 2000 / 35806
Done 2500 / 35806
Done 3000 / 35806
Done 3500 / 35806
Done 4000 / 35806
Done 4500 / 35806
Done 5000 / 35806
Done 5500 / 35806
Done 6000 / 35806
Done 6500 / 35806
Done 7000 / 35806
Done 7500 / 35806
Done 8000 / 35806
Done 8500 / 35806
Done 9000 / 35806
Done 9500 / 35806
Done 10000 / 35806
Done 10500 / 35806
Done 11000 / 35806
Done 11500 / 35806
Done 12000 / 35806
Done 12500 / 35806
Done 13000 / 35806
Done 13500 / 35806
Done 14000 / 35806
Done 14500 / 35806
Done 15000 / 35806
Done 15500 / 35806
Done 16000 / 35806
Done 16500 / 35806
Done 17000 / 35806
Done 17500 / 35806
Done 18000 / 35806
Done 18500 / 35806
Done 19000 / 35806
Done 19500 / 35806
Done 20000 / 35806
Done 20500 / 35806
Done 21000 / 35806
Done 21500 / 35806
Done 22000 / 35806
Done 22500 / 35806
Done 23000 / 35806
Done 23500 / 35806
Done 24000 / 35806
Done 24500 / 35806
Done 25000 / 35806
Done 25500 / 35806
Done 26000 / 35806
Done 26500 / 35806
Done 27000 / 

In [ ]:
35,806 genes
        │
        ▼
───────────────────────────────
Chunk 1 (500 genes)
        │
        ▼
500 × 384 matrix
        │
        ▼
Stored in all_embeddings
───────────────────────────────

Chunk 2 (500 genes)
        │
        ▼
500 × 384 matrix
        │
        ▼
Stored in all_embeddings

...

Final chunk (306 genes)
        │
        ▼
306 × 384 matrix

In [40]:
#combine all chunks
embeddings = np.vstack(all_embeddings)

print(embeddings.shape)

(35806, 384)


In [ ]:
#vstack means Vertical Stack.

In [41]:
#verify
print(type(embeddings))
print(embeddings.shape)
print(embeddings.dtype)

<class 'numpy.ndarray'>
(35806, 384)
float32


In [42]:
np.save("../data/rice_embeddings.npy", embeddings)

print("Embeddings saved successfully!")

Embeddings saved successfully!


In [ ]:
But embeddings are NumPy arrays, not tables.

NumPy has its own very fast binary format:

.npy

In [44]:
#build faiss index 
import faiss
import numpy as np

embeddings= np.load("../data/rice_embeddings.npy").astype("float32")

In [45]:
index= faiss.IndexFlatL2(384)
index.add(embeddings)
print(index.ntotal)

35806


In [ ]:
#L2 is the Euclidean distance.

In [ ]:
#Instead of checking all 35k vectors manually every time, FAISS is optimized for finding the closest ones efficiently.

In [49]:
query= "photosynthesis protein involved in chloroplast"
query_embedding= model.encode(query).astype("float32")

In [50]:
D, I = index.search(
    query_embedding.reshape(1, -1),
    k=5
)

In [ ]:
Your query embedding currently has shape:

(384,)

FAISS expects:

(number_of_queries, dimensions)

Even if you're asking only one question, it expects a 2D matrix.

So

reshape(1, -1)

changes:384,)

into

(1,384)

In [51]:
df.iloc[I[0]][
    [
        "Gene_ID",
        "GO_Term",
        "Description"
    ]
]

,Gene_ID,GO_Term,Description
18987,Os03g0792400,proteolysis; chloroplast; chloroplast organiza...,No description
12436,Os02g0257300,chloroplast thylakoid membrane; chloroplast; p...,No description
24152,Os05g0393400,chloroplast mRNA processing; leaf development;...,No description
25609,Os06g0107700,oxidoreductase activity; photosynthesis; chlor...,"ferredoxin--NADP reductase, leaf isozyme 1, ch..."
14558,Os02g0744000,photosynthetic electron transport in photosyst...,No description


In [53]:
def search_gene(query, k=5):

    q = model.encode(query).astype("float32")

    D, I = index.search(
        q.reshape(1,-1),
        k
    )

    return df.iloc[I[0]][
        [
            "Gene_ID",
            "Description",
            "GO_Term"
        ]
    ]

In [ ]:
#I stands for index. these are the row numbers inside the dataframe
#D means distance. smaller distance=more similarity

In [54]:
search_gene("drought resistance")

,Gene_ID,Description,GO_Term
1933,Os01g0526100,No description,RNA binding; nucleic acid binding; mRNA cis sp...
3636,Os01g0798500,No description,response to photooxidative stress; photosystem...
8166,Os11g0453900,No description,response to water; response to abscisic acid; ...
8162,Os11g0451700,No description,response to water; response to abscisic acid; ...
8170,Os11g0454300,No description,response to water; response to abscisic acid; ...


In [55]:
search_gene("photosynthesis")

,Gene_ID,Description,GO_Term
14558,Os02g0744000,No description,photosynthetic electron transport in photosyst...
4143,Os01g0869800,No description,photosynthesis; photosystem II; chloroplast th...
33695,Os08g0566600,No description,photosynthetic electron transport in photosyst...
21037,Os04g0473150,No description,photosynthetic electron transport in photosyst...
34088,Os09g0306650,No description,photosynthetic electron transport in photosyst...


In [56]:
search_gene("DNA repair")

,Gene_ID,Description,GO_Term
3361,Os01g0757800,No description,damaged DNA binding; DNA repair; transferase a...
31318,Os08g0107600,No description,nucleus; DNA repair; DNA damage response; liga...
22793,Os05g0111000,No description,nucleic acid binding; DNA binding; DNA replica...
20548,Os04g0401800,No description,nucleus; DNA repair; DNA damage response; doub...
28200,Os06g0693300,No description,DNA binding; nucleus; DNA repair; DNA damage r...


In [57]:
while True:

    query = input("Ask PlantGPT: ")

    if query.lower() == "exit":
        break

    print(search_gene(query))

Ask PlantGPT:  photosynthesis)


            Gene_ID     Description  \
4143   Os01g0869800  No description   
21396  Os04g0523000  No description   
28564  Os07g0105600  No description   
14698  Os02g0764500  No description   
33695  Os08g0566600  No description   

                                                 GO_Term  
4143   photosynthesis; photosystem II; chloroplast th...  
21396  photosystem II; photosynthesis; photosystem II...  
28564  photosystem II; photosynthesis; photosystem II...  
14698  membrane; photosynthesis, light harvesting; ph...  
33695  photosynthetic electron transport in photosyst...  


Ask PlantGPT:  exit


In [58]:
#save the faiss index
faiss.write_index(index, "../data/rice_faiss.index")

print("FAISS index saved!")

FAISS index saved!


In [59]:
print(search_gene("DNA repair"))

            Gene_ID     Description  \
3361   Os01g0757800  No description   
31318  Os08g0107600  No description   
22793  Os05g0111000  No description   
20548  Os04g0401800  No description   
28200  Os06g0693300  No description   

                                                 GO_Term  
3361   damaged DNA binding; DNA repair; transferase a...  
31318  nucleus; DNA repair; DNA damage response; liga...  
22793  nucleic acid binding; DNA binding; DNA replica...  
20548  nucleus; DNA repair; DNA damage response; doub...  
28200  DNA binding; nucleus; DNA repair; DNA damage r...  
